### ***RAG Application***
#### **End-to-End Rag Application**

In [1]:
### load the Environment Variables

from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
### Check/Validate the path to load the files from the Document
import os

path = "../kubernetes"

if os.path.exists(path):
    print("Path is valid")
else:
    print("Path is not valid")

Path is valid


In [3]:
#load the documents using DirectoryLoader 

from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader

loader = DirectoryLoader(
    path,
    glob = "*.pdf",
    loader_cls=PyMuPDFLoader
)

documents = loader.load()

print("Number Of Documents:",len(documents))

C:\Users\mukko\AppData\Local\Temp\ipykernel_9260\894974917.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader


Number Of Documents: 3983


In [4]:
### create a chunks using splitter

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)

print("Number Of Chunks:",len(chunks))

Number Of Chunks: 12694


In [5]:
###BM25 Retriever
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k=10

In [6]:
####Initalize the embedding model

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model = "BAAI/bge-large-en-v1.5",
    model_kwargs = {"device":"cpu"},
    encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [7]:
### load the vcctors from Chroma DB
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name="kubernetes_rag",
    embedding_function=embedding_model
)


In [8]:
#### retriever for similarity search.
vector_retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":10}
)

In [9]:
### Hybrid Search
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever,bm25_retriever],
    weights = [0.7,0.3]
)

In [10]:
### Lets test with hybrid retriever and how many candidates are retrieved
docs = hybrid_retriever.invoke("What is Kubernetes Deployment?")
print(len(docs))

20


In [11]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [12]:
### Get top 5 final document after reranking

def retrieve_and_rerank(query, k=5):

    retrieved_docs = hybrid_retriever.invoke(query)


    pairs = [[query,doc.page_content] for doc in retrieved_docs]

    scores = reranker.predict(pairs)

    ##zip the scores and retrieved order the documents based on score from highest to lowest

    ranked_docs = sorted(zip(retrieved_docs,scores),key = lambda x:x[1],reverse=True)

    ## Get top k documents using ranked_docs and k value
    top_docs = [doc for (doc,score) in ranked_docs[:k]]

    return top_docs

In [13]:
### Format without unnecessary context

def build_context(documents):

    context = ""

    for i,doc in enumerate(documents,start=1):
        source = doc.metadata.get("source")
        source = source.replace("\\","/")
        source = source.split("/")[-1]
        #print(source)
        context+=f"""
        
    Source {source}
    Page: {doc.metadata.get("page")}
    Content: {doc.page_content}
        """
    return context



In [14]:
### Design a prompt
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
You are a question-answering assistant.

Answer the question using ONLY the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the context does not contain the answer, say:
   "I don't have information based on the provided documents."
4. Keep the answer concise and directly relevant.
5. List the sources and page numbers used at the very end of your response.
6. Format the sources strictly like this:
[Source: filename, Page: number]

Context:
{context}

Question:
{question}

Answer:
""")

In [15]:
### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-safeguard-20b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'Safety GPT OSS 20B', 'release_date': '2025-03-05', 'last_updated': '2025-03-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000184A79EA7E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000184A78D6D20>, model_name='openai/gpt-oss-safeguard-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

#### ***Step 1: Deduplicate the reranked results before building the LLM context.***

##### **In Deduplication what if page/Documents splits into three chunks, these chunks points to the same page right but with different content may be it's useful we have to handle it otherwise it may lead to hallucination.**

In [16]:
query = "What is Kubernetes Deployment?"

top_5_chunks = retrieve_and_rerank(query)

for i,chunk in enumerate(top_5_chunks,start=1):
    print(f"------ Document {i} --------")
    print("Page:",chunk.metadata.get("page"))

------ Document 1 --------
Page: 9
------ Document 2 --------
Page: 5
------ Document 3 --------
Page: 638
------ Document 4 --------
Page: 6
------ Document 5 --------
Page: 12


In [17]:
from langchain_core.tracers import LangChainTracer

custom_tracer = LangChainTracer(project_name="Generation Evaluation For Kubernetes RAG.")

In [18]:
config = {"run_name":"Generation Evaluation","callbacks":[custom_tracer]}

In [23]:
def final_rag_response(query):

    top_5_docs = retrieve_and_rerank(query)
    context = build_context(top_5_docs)

    #print("Context:",context)

    messages = prompt.invoke({"question":query,"context":context})

    response = llm.invoke(messages,config=config).content

    return response

In [24]:
test_queries = [
    # 1. Direct factual
    "What is a Kubernetes Deployment?",

    # 2. Conceptual
    "Why is a Pod considered the smallest deployable unit in Kubernetes?",

    # 3. Comparison
    "What is the difference between a Pod and a Deployment?",

    # 4. Relationship
    "How are Deployments, ReplicaSets, and Pods related?",

    # 5. Process
    "How does a Deployment manage ReplicaSets and Pods during an update?",

    # 6. Multi-part
    "What is a Pod, what resources do its containers share, and how do they communicate?",

    # 7. Unsupported / negative
    "How does a Python decorator work?"
]

for query in test_queries:
    response = final_rag_response(query)
    print("="*60)
    print("Query:",query)
    print("="*60)
    print("Response:",response)

Query: What is a Kubernetes Deployment?
Response: A Kubernetes Deployment is an object that represents an application running on a cluster.  
When you create it, you specify a desired state (e.g., number of replicas). The control plane then launches that many Pods, continuously monitors them, and automatically replaces or restarts Pods if a node fails or a container terminates—providing self‑healing and scaling.  

[Source: Concepts.pdf, Page: 5]  
[Source: Tutorials.pdf, Page: 9]  
[Source: Tutorials.pdf, Page: 2]
Query: Why is a Pod considered the smallest deployable unit in Kubernetes?
Response: A Pod is the smallest deployable unit in Kubernetes because it is the most basic object that you can create, manage, and schedule on a cluster.  
- It is defined as a group of one or more containers that share storage, networking, and a run‑time specification.  
- All containers in a Pod are co‑located and co‑scheduled, and once a Pod is created you cannot add more containers to it—new Pods 